# Lesson 0 — A binary book has two bid ladders and no asks

Kalshi publishes resting YES bids and resting NO bids. The offer you would trade against is a reading of the opposite ladder, so the sum of the two asks is one dollar plus the spread and is never below a dollar.

**The rule.** `yes_ask + no_ask = (1 − no_bid) + (1 − yes_bid) = 1 + spread`

**When it holds.** Always, on any single snapshot of one market. It is an identity in the definitions, not a property of a liquid book.

**When it fails.** Never as arithmetic — but it appears to fail when the two ladders are read at different instants. A sum below a dollar means a torn snapshot, and a bot that trades on it is trading on its own latency.

| | |
|---|---|
| Lesson id | `book` |
| Pane it appears on | `books` (panes carry more than one lesson) |
| Code it is about | `modules/coherence/kernel/book.py` |
| Tests that go red if it stops being true | `tests/test_coherence_lesson_0.py` |
| Pane shipped | yes |

Every cell below runs against the real kernel. Nothing here is a re-implementation:
a number this notebook prints is the number the engine would produce for the same
input. The recorded Kalshi payloads come from `tests/fixtures/coherence/`.

In [ ]:
import json
import sys
from decimal import Decimal
from pathlib import Path

# This notebook lives in notebooks/coherence_lab/ and imports the kernel two
# levels up. Found by walking upward rather than by counting parents, so the
# notebook runs from its own directory or from Part2_Infrastructure.
HERE = Path.cwd().resolve()
ROOT = next((path for path in (HERE, *HERE.parents) if (path / "modules" / "coherence" / "kernel").is_dir()), None)
if ROOT is None:
    raise SystemExit(f"no coherence kernel above {HERE}: open this notebook from inside Part2_Infrastructure")
sys.path.insert(0, str(ROOT))

FIXTURES = ROOT / "tests" / "fixtures" / "coherence"


def fixture(name: str) -> dict:
    """One recorded Kalshi response, envelope and all, exactly as it was sent.

    These are captures, not mocks. Where a number below looks odd it is because
    the exchange quoted it, and `tools/capture_kalshi_fixtures.py` re-records
    them.
    """
    return json.loads((FIXTURES / f"{name}.json").read_text(encoding="utf-8"))


print(f"kernel root       {ROOT}")
print(f"recorded fixtures {FIXTURES.is_dir()}")

## 1. The two ladders, as the venue sent them

In [ ]:
from modules.coherence.kernel.book import Book, Level, lesson_zero_identity, parse_orderbook

recorded = fixture("orderbook_two_sided")
ticker = recorded["source"].split("/markets/")[1].split("/")[0]
book = parse_orderbook(ticker, recorded["body"]["orderbook_fp"])
captured = recorded["captured_at"]

print(f"{ticker}, recorded {captured}")
print(f"  {len(book.yes_bids)} resting YES bids")
print(f"  {len(book.no_bids)} resting NO bids")
print("  0 asks, on either side. The venue publishes none.")
print()
print("  top of each ladder, kept as sent (ascending, best last):")
print(f"    YES {book.yes_bids[-1].price} for {book.yes_bids[-1].size} contracts")
print(f"    NO  {book.no_bids[-1].price} for {book.no_bids[-1].size} contracts")

## 2. The asks nobody published

In [ ]:
print("Every ask below is a reading of the opposite ladder, not a queue of its own:")
print(f"  best YES bid   {book.best_yes_bid}   a real resting order")
print(f"  best NO bid    {book.best_no_bid}   a real resting order")
print(f"  best YES ask   {book.best_yes_ask}   = 1 - no_bid")
print(f"  best NO ask    {book.best_no_ask}   = 1 - yes_bid")
print(f"  spread         {book.spread}")

## 3. The identity, on this book and on ten thousand others

In [ ]:
left, right = lesson_zero_identity(book)
print(f"  yes_ask + no_ask = {left}")
print(f"  1 + spread       = {right}")
print(f"  identical        : {left == right}")
print()

# Not a property of this book. Sweep every one-cent pair of bids and it holds.
checked = 0
broken = 0
for yes_cents in range(1, 100):
    for no_cents in range(1, 100):
        probe = Book(
            ticker="PROBE",
            yes_bids=(Level(Decimal(yes_cents) / 100, 10_000),),
            no_bids=(Level(Decimal(no_cents) / 100, 10_000),),
        )
        pair = lesson_zero_identity(probe)
        checked += 1
        if pair is None or pair[0] != pair[1]:
            broken += 1
print(f"  {checked} synthetic books swept, {broken} broke the identity")

## 4. Why 'buy both sides under a dollar' is unreachable

In [ ]:
pairs = 0
under_a_dollar = 0
cheapest = None
for yes_level in book.asks("yes"):
    for no_level in book.asks("no"):
        total = yes_level.price + no_level.price
        pairs += 1
        if total < Decimal(1):
            under_a_dollar += 1
        cheapest = total if cheapest is None else min(cheapest, total)

print(f"  {pairs} (YES ask, NO ask) pairs across the whole recorded book")
print(f"  cheapest pair sums to     {cheapest}")
print(f"  pairs under a dollar      {under_a_dollar}")
print(f"  1 + spread                {Decimal(1) + book.spread}")
print()
print("  The cheapest pair IS one plus the spread, because both asks are read off the")
print("  best bid on the other side. Every deeper level is worse. The branch that hunts")
print("  for a sum below a dollar cannot fire on a single snapshot: it is dead code.")

## 5. The only way it appears to fail: a torn snapshot

In [ ]:
# The one way the sum falls below a dollar: the two ladders read at different
# instants. Shift the NO ladder by the spread plus a tick and read the YES
# ladder as it was a moment ago.
shift = book.spread + Decimal("0.0100")
torn = Book(
    ticker=book.ticker,
    yes_bids=book.yes_bids,
    no_bids=(Level(book.best_no_bid + shift, 10_000),),
)
print(f"  ladders {shift} apart in time")
print(f"  yes_ask {torn.best_yes_ask} + no_ask {torn.best_no_ask} = {torn.best_yes_ask + torn.best_no_ask}")
print(f"  and the implied spread is {torn.spread}, which no single instant can be")
print()
print("  A bot that fires here is trading its own latency, not the market's prices.")